# Day 3 · LLM 모의 실기

기본 LLM01~14: 90점 · 확장 LLM15~16: 10점. 총 16문항 / 100점.
기본 문제는 Chapter 2의 비교 이전, Chapter 3~6에 대응합니다. 확장은 Chapter 7 지시 학습·DPO입니다.
권장 첫 풀이 시간은 기본 150~210분, 확장 25~40분의 가안입니다. 초회 학습·복습 시간은 별도입니다.
평가 축: 데이터 구성 / 주요 레이어 / Tensor 흐름 / 학습 대상과 학습 작업. 단어장 비교 이후 내용은 제외했습니다.

**연습용 모의시험입니다. 실제 기출·공식 배점·공식 제한 시간이 아닙니다.**
근거: [인증시험 출제 포인트 PDF](../../AI인증시험_출제포인트.pdf) 1쪽과 문제별 교안 셀.
셀 번호는 Markdown을 포함하여 맨 위부터 1로 센 번호입니다.

## 푸는 방법

1. 이 파일을 복사하여 `-submission.ipynb` 이름으로 저장합니다. 기존 실습 답안은 수정하지 않습니다.
2. `env_aias_test` 커널을 선택하고 아래 **공통 준비 셀**부터 실행합니다.
3. 문제의 `????`를 코드로 바꾸거나 `raise NotImplementedError` 자리에 필요한 몇 줄을 씁니다.
`????`는 의도적인 빈칸이므로 수정 전 해당 답안 셀은 SyntaxError가 납니다.
4. 답안 셀 → 바로 아래 확인 셀 순서로 실행합니다. 확인 셀은 수정하지 않습니다.
각 문제는 공통 준비 셀만 있으면 독립적으로 풀 수 있습니다.
5. 이유·shape 답변도 작성하고, 끝나면 **Restart Kernel → Run All**로 재검사합니다.
못 푼 문제는 번호를 기록하고 다음 문제로 이동할 수 있습니다.

데이터·작은 모델·인자는 제공됩니다. 인터넷 접속, 파일 다운로드, 설치, GPU, 장시간 학습은 필요하지 않습니다.
예제 입력의 숫자나 배치 크기를 답안에 고정하지 말고 전달된 인자를 사용하세요.
확인 셀 통과는 공개 사례의 동작 확인이며 전체 점수·개념 숙달을 보장하지 않습니다.
문제당 점수는 코드 요구 충족 80%, 이유·shape 설명 20%로 채점할 연습용 기준입니다.
정답 코드는 포함하지 않았습니다. 막힌 문제 번호를 알려주면 힌트부터 제공합니다.


## 공통 준비 · 제공 코드 / 수정하지 마세요


In [ ]:
import math
import copy
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
torch.manual_seed(123)
torch.set_num_threads(2)
device = torch.device("cpu")
print("PyTorch:", torch.__version__, "| device:", device)

class TinySequenceModel(nn.Module):
    # 문제 독립 실행용 작은 대체 모델. 실제 GPT 성능을 재현하지 않습니다.
    def __init__(self, vocab=16, dim=8, out=16):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim)
        self.head = nn.Linear(dim, out)
    def forward(self, ids):
        return self.head(self.emb(ids))


## LLM01 · 문자열 → 다음 토큰 Dataset → 배치 — 8점

**출처:** [Chapter_2_Exercise_Dataset.ipynb](../../llm_hands_on/Chapter_2_Exercise_Dataset.ipynb) 셀 1~4

**평가 축: 목적에 맞는 데이터 구성.** 주어진 tokenizer로 문자열을 토큰화하고, 길이 `max_length`인 입력과 한 칸 뒤 정답을 `stride` 간격으로 만드세요.
`encode_text`는 ID 리스트, Dataset은 `torch.long` 텐서 쌍을 반환합니다. 완전한 입력·정답 쌍만 포함하세요.
아래 TinyTokenizer는 다운로드 없는 API 연습용 대역으로 **BPE 구현이 아닙니다**. 실제 교안에서는 이미 준비된 GPT-2 tokenizer가 이 자리에 들어갑니다.
예: `a b c d e` → 길이 4 입력 `a b c d`, 정답 `b c d e`.

**짧은 설명 문제:** 입력 길이보다 토큰이 한 개 더 필요한 이유와 마지막 배치 크기를 설명하세요.


In [ ]:
class TinyTokenizer:
    words = ["a", "b", "c", "d", "e", "f", "g", "h", "i"]
    def encode(self, text): return [self.words.index(w) for w in text.split()]
    def decode(self, ids): return " ".join(self.words[i] for i in ids)

def encode_text(text, tokenizer):
    return tokenizer.????(text)

class NextTokenDataset(Dataset):
    def __init__(self, token_ids, max_length, stride):
        self.pairs = []
        for i in range(0, len(token_ids) - max_length, stride):
            x = token_ids[????:????]
            y = token_ids[????:????]
            self.pairs.append((torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)))
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx): return self.pairs[????]

def make_loader(dataset, batch_size, drop_last):
    return DataLoader(????, batch_size=batch_size, shuffle=False,
                      drop_last=drop_last, num_workers=0)


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
tokenizer = TinyTokenizer()
ids = encode_text("a b c d e f g h i", tokenizer)
assert tokenizer.decode(ids) == "a b c d e f g h i"
assert encode_text("c d", tokenizer) == [2,3]
ds = NextTokenDataset(ids, 4, 2)
assert len(ds) == 3
assert ds[1][0].tolist() == [2,3,4,5] and ds[1][1].tolist() == [3,4,5,6]
assert ds[0][0].dtype == torch.long
assert [x.shape[0] for x,y in make_loader(ds, 2, False)] == [2,1]
assert [x.shape[0] for x,y in make_loader(ds, 2, True)] == [2]
assert len(NextTokenDataset([0,1,2,3], 3, 1)) == 1
print("LLM01 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM02 · 토큰 임베딩과 위치 임베딩 — 6점

**출처:** [Chapter_2_Exercise_Dataset.ipynb](../../llm_hands_on/Chapter_2_Exercise_Dataset.ipynb) 셀 6~7 · [Chapter_4_Excercise_GPT.ipynb](../../llm_hands_on/Chapter_4_Excercise_GPT.ipynb) 셀 4

**평가 축: 레이어·Tensor 흐름.** `(B,T)` 정수 ID를 `(B,T,D)`로 바꾸세요. 토큰 표는 `(vocab_size,D)`, 위치 표는 `(context_length,D)`입니다.
위치는 각 문장에서 0부터 T-1까지입니다. 입력 ID와 같은 device에 위치 ID를 만드세요. T는 context_length 이하입니다.

**짧은 설명 문제:** vocab_size와 입력 토큰 개수 T는 왜 다를 수 있나요?


In [ ]:
class TokenPosition(nn.Module):
    def __init__(self, vocab_size, context_length, dim):
        super().__init__()
        self.tok = nn.Embedding(????, ????)
        self.pos = nn.Embedding(????, ????)
    def forward(self, ids):
        positions = torch.arange(????, device=ids.device)
        return self.tok(????) + self.pos(????)


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
m = TokenPosition(11, 7, 4)
with torch.no_grad():
    m.tok.weight.copy_(torch.arange(44).reshape(11,4))
    m.pos.weight.copy_(torch.arange(7).unsqueeze(1).expand(7,4) * 100)
ids = torch.tensor([[2,3,2], [1,0,4]])
y = m(ids)
assert y.shape == (2,3,4)
torch.testing.assert_close(y[0,2], torch.tensor([208.,209.,210.,211.]))
assert m(torch.tensor([[1,2]])).shape == (1,2,4)
print("LLM02 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM03 · Causal Attention 클래스 빈칸 — 10점

**출처:** [Chapter_3_Excercise_Attention.ipynb](../../llm_hands_on/Chapter_3_Excercise_Attention.ipynb) 셀 1~2

**평가 축: 주요 레이어·Tensor 흐름.** 입력 `(B,T,d_in)`에서 문맥 벡터 `(B,T,d_out)`와 가중치 `(B,T,T)`를 반환하세요.
현재·과거만 허용하고 미래 위치는 softmax 이전에 가립니다. 마스크는 학습 파라미터가 아닌 buffer로 등록합니다.
채점용으로 교안 반환값에 weights를 추가했습니다. dropout은 이 문제에서 생략합니다.

**짧은 설명 문제:** Q·K 비교 점수와 V를 섞은 결과의 역할 차이를 한국어로 설명하세요.


In [ ]:
class ExamCausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length):
        super().__init__()
        self.W_query = nn.Linear(????, ????, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=????).bool())
    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.W_query(x), self.W_key(x), self.W_value(x)
        scores = ???? @ k.transpose(????, ????)
        scores = scores.masked_fill(self.mask[:T,:T], ????)
        weights = torch.softmax(scores / (k.shape[-1] ** 0.5), dim=????)
        context = weights @ ????
        return context, weights


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
m = ExamCausalAttention(3, 2, 8)
x = torch.randn(2,5,3)
c,w = m(x)
assert c.shape == (2,5,2) and w.shape == (2,5,5)
torch.testing.assert_close(w.sum(-1), torch.ones(2,5))
assert torch.count_nonzero(w.triu(1)) == 0
torch.testing.assert_close(c[:,0], m.W_value(x)[:,0])
changed = x.clone(); changed[:,3:] += 100
torch.testing.assert_close(m(changed)[0][:,:3], c[:,:3])
assert 'mask' in m.state_dict() and 'mask' not in dict(m.named_parameters())
assert m(torch.randn(1,2,3))[0].shape == (1,2,2)
numeric=ExamCausalAttention(2,2,2)
with torch.no_grad():
    for layer in [numeric.W_query,numeric.W_key,numeric.W_value]: layer.weight.copy_(torch.eye(2))
sample=torch.tensor([[[1.,0.],[1.,2.]]])
context,weights=numeric(sample)
expected=torch.softmax(torch.tensor([1.,5.])/math.sqrt(2),dim=0)
torch.testing.assert_close(weights[0,1],expected)
torch.testing.assert_close(context[0,1],expected @ sample[0])
print("LLM03 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM04 · 멀티헤드 분리·결합 — 6점

**출처:** [Chapter_3_Excercise_Attention.ipynb](../../llm_hands_on/Chapter_3_Excercise_Attention.ipynb) 셀 4

**평가 축: Tensor 흐름.** 투영이 끝난 Q/K/V `(B,T,D)`를 H개 헤드로 나누고, 각 헤드에 causal attention을 적용하세요.
D는 H로 나누어떨어집니다. 결합 결과 `(B,T,D)`, 가중치 `(B,H,T,T)`를 반환합니다.
투영·dropout은 문제 외부에서 처리된 것으로 가정합니다.

**짧은 설명 문제:** B=2,T=6,D=8,H=4일 때 헤드별 Q와 scores의 shape를 적으세요.


In [ ]:
def split_heads(x, num_heads):
    b,t,d = x.shape
    return x.reshape(b,t,num_heads,????).transpose(????, ????)

def multihead_context(q, k, v, num_heads):
    b,t,d = q.shape
    qh,kh,vh = [split_heads(z,num_heads) for z in (q,k,v)]
    scores = qh @ kh.transpose(????, ????)
    mask = torch.triu(torch.ones(t,t,device=q.device,dtype=torch.bool),diagonal=1)
    weights = torch.softmax(scores.masked_fill(mask,-torch.inf) / math.sqrt(d // num_heads),dim=-1)
    heads = weights @ vh
    out = heads.transpose(????, ????).contiguous().view(b,t,d)
    return out, weights


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
qv = torch.zeros(2,3,8)
vv = torch.arange(48,dtype=torch.float32).reshape(2,3,8)
out,w = multihead_context(qv,qv,vv,4)
assert w.shape == (2,4,3,3) and out.shape == (2,3,8)
torch.testing.assert_close(out[:,1], vv[:,:2].mean(1))
for h in [1,2,4]:
    assert multihead_context(qv,qv,vv,h)[0].shape == vv.shape
probe = torch.arange(24).reshape(1,3,8)
assert split_heads(probe,2)[0,1,0].tolist() == [4,5,6,7]
print("LLM04 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM05 · LayerNorm·FeedForward — 6점

**출처:** [Chapter_4_Excercise_GPT.ipynb](../../llm_hands_on/Chapter_4_Excercise_GPT.ipynb) 셀 3

**평가 축: 레이어·학습 대상.** 각 토큰의 마지막 축만 평균·분산으로 정규화하세요. 분산은 모집단 분산(`unbiased=False`)입니다.
`scale`, `shift`는 학습되도록 등록합니다. FeedForward는 D→4D→D, 활성화는 GELU(tanh 근사)를 사용하세요.

**짧은 설명 문제:** 정규화 통계량 mean/var와 scale/shift 중 optimizer가 갱신하는 것은 무엇인가요?


In [ ]:
class ExamLayerNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.????(torch.ones(dim))
        self.shift = nn.Parameter(torch.zeros(dim))
    def forward(self, x):
        mean = x.mean(dim=????, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - ????) / torch.sqrt(???? + self.eps) + self.shift

def make_ff(dim):
    return nn.Sequential(nn.Linear(dim, ????), nn.GELU(approximate="tanh"), nn.Linear(????, dim))


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
x = torch.randn(2,3,4)
norm = ExamLayerNorm(4)
torch.testing.assert_close(norm(x), F.layer_norm(x,(4,)))
assert set(dict(norm.named_parameters())) == {'scale','shift'}
ff = make_ff(4)
assert ff[0].out_features == 16 and ff[2].in_features == 16
assert ff(x).shape == x.shape
norm(x).sum().backward()
assert norm.shift.grad is not None
print("LLM05 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM06 · TransformerBlock의 residual 연결 — 6점

**출처:** [Chapter_4_Excercise_GPT.ipynb](../../llm_hands_on/Chapter_4_Excercise_GPT.ipynb) 셀 4

**평가 축: 레이어 연결.** 생성자에서 제공한 모듈을 사용해 `forward`의 빠진 구간을 완성하세요.
순서는 Norm1→Attention→Dropout→입력 더하기, Norm2→FF→Dropout→첫 블록 결과 더하기입니다.
모든 모듈은 `(B,T,D)` shape를 유지합니다. `att`는 텐서 하나를 반환한다고 가정합니다.

**짧은 설명 문제:** 두 번째 shortcut으로 최초 입력을 다시 쓰면 왜 다른 모델이 되나요?


In [ ]:
class ExamBlock(nn.Module):
    def __init__(self, att, ff, norm1, norm2, drop):
        super().__init__()
        self.att, self.ff = att, ff
        self.norm1, self.norm2, self.drop = norm1, norm2, drop
    def forward(self, x):
        # TODO: 제공된 모듈을 연결하고 두 residual을 구현하세요.
        raise NotImplementedError("forward의 두 블록을 완성하세요")


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
class Times(nn.Module):
    def __init__(self,n): super().__init__(); self.n=n
    def forward(self,x): return self.n*x
m = ExamBlock(Times(2),Times(3),Times(4),Times(5),nn.Identity())
x = torch.randn(2,3,4)
torch.testing.assert_close(m(x), x * 144)
zero = Times(0)
identity = ExamBlock(zero,zero,nn.Identity(),nn.Identity(),nn.Identity())
torch.testing.assert_close(identity(x),x)
print("LLM06 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM07 · GPT 출력까지 연결 — 6점

**출처:** [Chapter_4_Excercise_GPT.ipynb](../../llm_hands_on/Chapter_4_Excercise_GPT.ipynb) 셀 4

**평가 축: 입력→출력 Tensor 흐름.** 주어진 blocks를 포함하여 ID→토큰·위치 임베딩→blocks→final_norm→어휘 logits를 연결하세요.
입력 `(B,T)` long, 출력 `(B,T,vocab_size)` float입니다. 출력에 softmax를 적용하지 마세요.
독립 검사를 위해 blocks를 인자로 주입합니다. 실제 교안에서는 TransformerBlock들을 쌓은 모듈입니다.

**짧은 설명 문제:** 출력 마지막 축이 임베딩 차원 D가 아니라 vocab_size인 이유를 설명하세요.


In [ ]:
class ExamGPT(nn.Module):
    def __init__(self, vocab_size, context_length, dim, blocks):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size,dim)
        self.pos_emb = nn.Embedding(context_length,dim)
        self.blocks = blocks
        self.final_norm = nn.LayerNorm(dim)
        self.out_head = nn.Linear(????, ????, bias=False)
    def forward(self, ids):
        pos = torch.arange(ids.shape[1],device=ids.device)
        x = self.tok_emb(ids) + ????
        x = self.????(x)
        x = self.final_norm(x)
        return self.????(x)


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
block = nn.Linear(6,6,bias=False)
model = ExamGPT(13,8,6,block)
ids = torch.tensor([[1,2,3],[4,5,6]])
y = model(ids)
assert y.shape == (2,3,13)
embedded = model.tok_emb(ids) + model.pos_emb(torch.arange(3))
torch.testing.assert_close(y,model.out_head(model.final_norm(block(embedded))))
assert model(torch.tensor([[1]])).shape == (1,1,13)
y.square().mean().backward()
assert model.tok_emb.weight.grad is not None and block.weight.grad is not None
print("LLM07 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM08 · 문맥 자르기와 greedy 생성 — 6점

**출처:** [Chapter_4_Excercise_GPT.ipynb](../../llm_hands_on/Chapter_4_Excercise_GPT.ipynb) 셀 5 · [Chapter_5_Excercise_Pretraining.ipynb](../../llm_hands_on/Chapter_5_Excercise_Pretraining.ipynb) 셀 4

**평가 축: 모델 활용·Tensor 흐름.** 마지막 context_size개 토큰만 모델에 넣고, 마지막 위치의 최대 logit ID를 하나씩 이어 붙이세요.
생성 전 `eval` 모드, 생성 중 gradient 기록 금지. EOS 처리는 이 문제에 없습니다.
입력 `(B,T)` → `(B,T+max_new_tokens)`. 원래 prefix는 그대로 보존합니다.

**짧은 설명 문제:** 전체 시퀀스 logits 중 마지막 위치만 사용하는 이유를 적으세요.


In [ ]:
def generate_greedy(model, ids, max_new_tokens, context_size):
    model.????()
    for _ in range(max_new_tokens):
        cropped = ids[:, ????]
        with torch.????():
            logits = model(cropped)
        last = logits[:, ????, :]
        next_id = last.????(dim=-1,keepdim=True)
        ids = torch.cat((ids,next_id),dim=????)
    return ids


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
class IncrementModel(nn.Module):
    def __init__(self): super().__init__(); self.seen=[]
    def forward(self,ids):
        assert not torch.is_grad_enabled() and not self.training
        self.seen.append(ids.clone())
        return F.one_hot((ids+1)%10,10).float()*10
m = IncrementModel()
result = generate_greedy(m,torch.tensor([[1,2,3],[4,5,6]]),3,2)
assert result.tolist() == [[1,2,3,4,5,6],[4,5,6,7,8,9]]
assert m.seen[0].tolist() == [[2,3],[5,6]]
assert all(t.shape == (2,2) for t in m.seen)
assert generate_greedy(m,torch.tensor([[1]]),0,2).tolist() == [[1]]
print("LLM08 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM09 · 토큰별 Cross Entropy — 6점

**출처:** [Chapter_5_Excercise_Pretraining.ipynb](../../llm_hands_on/Chapter_5_Excercise_Pretraining.ipynb) 셀 2

**평가 축: 예측·정답→학습.** 모델이 `(B,T,V)` raw logits를 반환합니다. 정답은 이미 한 칸 shift된 `(B,T)` long입니다.
모든 위치의 평균 cross entropy를 scalar로 반환하세요. 여기에서 정답을 다시 shift하지 않습니다.

**짧은 설명 문제:** B=2,T=3,V=7에서 손실함수에 전달되는 logits·target shape를 적으세요.


In [ ]:
def token_loss(model, inputs, targets, device):
    inputs,targets = inputs.to(device),targets.to(device)
    logits = model(inputs)
    return F.????(logits.flatten(????, ????),targets.????())


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
model = TinySequenceModel(vocab=7,dim=5,out=7)
ids = torch.tensor([[0,1,2],[3,4,5]])
target = torch.tensor([[1,2,3],[4,5,6]])
loss = token_loss(model,ids,target,device)
reference = F.cross_entropy(model(ids).transpose(1,2),target)
assert loss.ndim == 0 and loss.requires_grad
torch.testing.assert_close(loss,reference)
assert token_loss(model,ids[:1,:2],target[:1,:2],device).ndim == 0
print("LLM09 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM10 · 한 배치 학습 순서 완성 — 6점

**출처:** [Chapter_5_Excercise_Pretraining.ipynb](../../llm_hands_on/Chapter_5_Excercise_Pretraining.ipynb) 셀 3

**평가 축: 학습에 필요한 작업.** `loss_fn(model,inputs,targets)`가 gradient가 연결된 scalar loss를 반환합니다.
학습 모드 전환→이전 gradient 초기화→손실 계산→역전파→optimizer 갱신을 구현하고 Python float loss를 반환하세요.
다른 문제의 token_loss를 완성하지 않아도 아래 확인 셀에서 loss_fn을 제공합니다.

**짧은 설명 문제:** backward와 optimizer.step은 각각 무엇을 바꾸나요?


In [ ]:
def train_batch(model, optimizer, inputs, targets, loss_fn):
    # TODO: 순서에 맞는 약 6줄을 작성하세요.
    raise NotImplementedError("한 배치 학습 구현")


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
m = nn.Linear(2,2)
other = copy.deepcopy(m)
x = torch.tensor([[1.,2.],[-1.,1.]])
y = torch.tensor([0,1])
criterion = lambda model,a,b: F.cross_entropy(model(a),b)
opt = torch.optim.SGD(m.parameters(),lr=.1)
opt2 = torch.optim.SGD(other.parameters(),lr=.1)
m.eval()
for p in m.parameters(): p.grad = torch.ones_like(p)*100
before = m.weight.detach().clone()
result = train_batch(m,opt,x,y,criterion)
other.train(); opt2.zero_grad(); loss2=criterion(other,x,y); loss2.backward(); opt2.step()
assert isinstance(result,float) and m.training
assert not torch.equal(m.weight,before)
for p,r in zip(m.parameters(),other.parameters()): torch.testing.assert_close(p,r)
print("LLM10 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM11 · 분류용 자르기·패딩·정답 구성 — 6점

**출처:** [Chapter_6_Excercise_Finetuning_Classification.ipynb](../../llm_hands_on/Chapter_6_Excercise_Finetuning_Classification.ipynb) 셀 1 / SpamDataset

**평가 축: 목적에 맞는 데이터 구성.** 이미 토큰화한 가변 길이 sequences와 문장별 labels를 제공합니다.
앞 max_length개까지 자른 뒤 오른쪽을 pad_id로 채워 `(B,max_length)` long을 만드세요. 라벨은 `(B,)` long입니다.
입력 리스트는 수정하지 마세요. 예: `[2,3]`, 길이 4, pad=0 → `[2,3,0,0]`.

**짧은 설명 문제:** 다음 토큰 예측의 정답과 문장 분류 정답은 shape가 어떻게 다른가요?


In [ ]:
def classification_batch(sequences, labels, max_length, pad_id):
    rows = []
    for seq in sequences:
        row = seq[:????]
        row = row + [????] * (???? - len(row))
        rows.append(row)
    return torch.tensor(rows,dtype=????), torch.tensor(labels,dtype=????)


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
sequences = [[2,3],[4,5,6,7,8]]
before = copy.deepcopy(sequences)
x,y = classification_batch(sequences,[1,0],4,0)
assert x.tolist() == [[2,3,0,0],[4,5,6,7]] and y.tolist() == [1,0]
assert x.dtype == y.dtype == torch.long and y.shape == (2,)
assert sequences == before
assert classification_batch([[1]],[0],2,9)[0].tolist() == [[1,9]]
print("LLM11 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM12 · 학습 대상 동결·분류 head 교체 — 6점

**출처:** [Chapter_6_Excercise_Finetuning_Classification.ipynb](../../llm_hands_on/Chapter_6_Excercise_Finetuning_Classification.ipynb) 셀 1~2

**평가 축: 무엇을 학습하는가.** 먼저 모든 기존 파라미터를 동결하세요. out_head를 dim→num_classes의 새 Linear로 교체합니다.
마지막 trf_blocks 블록과 final_norm만 다시 학습 가능하게 하고 모델을 반환하세요. CPU 모델이 제공됩니다.
`classification_logits`는 교안 규칙대로 **패딩을 포함한 마지막 위치**의 logits `(B,C)`를 선택합니다.
마지막 실제 단어 위치를 찾는 방식과 혼동하지 마세요.

**짧은 설명 문제:** 새 head를 만들기 전에 동결하는 이유와 optimizer에 전달할 학습 대상들을 적으세요.


In [ ]:
def prepare_classifier(model, dim, num_classes):
    for p in model.parameters(): p.requires_grad = ????
    model.out_head = nn.Linear(????, ????)
    for p in model.trf_blocks[????].parameters(): p.requires_grad = True
    for p in model.final_norm.parameters(): p.requires_grad = ????
    return model

def classification_logits(sequence_logits):
    return sequence_logits[:, ????, :]


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
class ClassifierFixture(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb=nn.Embedding(11,4)
        self.trf_blocks=nn.Sequential(nn.Linear(4,4),nn.Linear(4,4))
        self.final_norm=nn.LayerNorm(4)
        self.out_head=nn.Linear(4,11)
m=prepare_classifier(ClassifierFixture(),4,3)
assert not m.tok_emb.weight.requires_grad
assert not any(p.requires_grad for p in m.trf_blocks[0].parameters())
assert all(p.requires_grad for p in m.trf_blocks[-1].parameters())
assert all(p.requires_grad for p in m.final_norm.parameters())
assert m.out_head.out_features == 3 and m.out_head.weight.requires_grad
z=torch.arange(30).reshape(2,5,3)
assert classification_logits(z).tolist() == [[12,13,14],[27,28,29]]
print("LLM12 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM13 · LoRA 부분 클래스 완성 — 6점

**출처:** [Chapter_6_Excercise_Finetuning_Classification_LoRA.ipynb](../../llm_hands_on/Chapter_6_Excercise_Finetuning_Classification_LoRA.ipynb) 셀 3~4

**평가 축: 학습 대상·레이어 구성.** 기존 Linear는 동결하고, `A(in_dim,rank)`, `B(rank,out_dim)`만 학습하게 만드세요.
출력은 기존 Linear 출력에 `(alpha/rank) × xAB`를 더합니다. B는 0으로 초기화합니다.
입력은 `(B,T,in_dim)`처럼 앞쪽 축이 있어도 동작해야 합니다.

**짧은 설명 문제:** Low-Rank Adaptation에서 기존 weight와 A/B 중 어떤 값이 갱신되나요?


In [ ]:
class ExamLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        for p in self.linear.parameters(): p.requires_grad = ????
        self.rank, self.alpha = rank,alpha
        self.A = nn.Parameter(torch.randn(linear.in_features,????)*.01)
        self.B = nn.Parameter(torch.zeros(????,linear.out_features))
    def forward(self,x):
        return self.linear(x) + (self.alpha / ????) * (x @ ???? @ ????)


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
m=ExamLoRA(nn.Linear(5,3),2,4)
x=torch.randn(2,4,5)
torch.testing.assert_close(m(x),m.linear(x))
assert sum(p.numel() for p in m.parameters() if p.requires_grad) == 16
with torch.no_grad(): m.B.fill_(.3)
torch.testing.assert_close(m(x),m.linear(x)+2*(x@m.A@m.B))
m(x).sum().backward()
assert m.linear.weight.grad is None and m.A.grad is not None and m.B.grad is not None
print("LLM13 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM14 · 분류 평가·샘플 수 기준 집계 — 6점

**출처:** [Chapter_6_Excercise_Finetuning_Classification.ipynb](../../llm_hands_on/Chapter_6_Excercise_Finetuning_Classification.ipynb) 셀 1 / calc_accuracy_loader

**평가 축: 모델 활용·평가 데이터 흐름.** 모델은 `(B,T,C)` logits를 반환합니다. 마지막 위치에서 클래스별 최대 점수를 선택하세요.
eval·no_grad를 적용하고, 전체 정답 개수 / 전체 샘플 수로 accuracy를 반환합니다.
loader는 비어 있지 않고 마지막 배치 크기가 다를 수 있습니다.

**짧은 설명 문제:** 첫 배치 정확도 1, 마지막 배치 정확도 0일 때 단순 평균 .5가 틀리는 예를 설명하세요.


In [ ]:
def evaluate_classifier(model, loader):
    model.????()
    correct,total = 0,0
    with torch.????():
        for ids,labels in loader:
            logits = model(ids)[:, -1, :]
            pred = logits.argmax(dim=????)
            correct += (pred == labels).sum().item()
            total += ????
    return correct / ????


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
class EvalFixture(nn.Module):
    def forward(self,ids):
        assert not self.training and not torch.is_grad_enabled()
        return F.one_hot(ids%2,2).float()*5
loader=DataLoader(TensorDataset(torch.tensor([[0],[1],[0]]),torch.tensor([0,1,1])),batch_size=2)
m=EvalFixture()
assert abs(evaluate_classifier(m,loader)-2/3) < 1e-6
print("LLM14 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM15 · 확장 · 지시 학습 collate — 5점

**출처:** [Chapter_7_Exercise_Follow_Instructions.ipynb](../../llm_hands_on/Chapter_7_Exercise_Follow_Instructions.ipynb) 셀 1 / custom_collate_fn

**평가 축: 의도에 맞는 데이터 구성.** 토큰화한 지시+응답 시퀀스를 제공합니다. 마지막 EOS를 하나 추가하고 배치 길이를 맞추세요.
`pad_id`를 EOS와 padding에 함께 사용합니다. 정답에서 **첫 EOS는 학습에 포함**, 그 뒤 padding만 -100으로 바꿉니다.
입력 시퀀스에는 pad_id가 원래 없다고 가정합니다. 출력 둘 다 `(B,배치 내 원본 최대 길이)` long입니다.
Chapter 7 범위를 함께 점검할 확장 문제이며, 공식 출제 비중이 확인된 것은 아닙니다.

**짧은 설명 문제:** 첫 EOS까지 -100으로 지우면 모델이 배우지 못하는 것은 무엇인가요?


In [ ]:
def instruction_collate(batch, pad_id=0, ignore_index=-100):
    length = max(len(row) for row in batch)+1
    xs,ys=[],[]
    for row in batch:
        padded = row + [pad_id] * (length-len(row))
        x=torch.tensor(padded[????],dtype=torch.long)
        y=torch.tensor(padded[????],dtype=torch.long)
        eos_positions=(y == pad_id).nonzero(as_tuple=True)[0]
        y[eos_positions[????:]]=ignore_index
        xs.append(x); ys.append(y)
    return torch.stack(xs),torch.stack(ys)


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
x,y=instruction_collate([[1,2,3],[4]],pad_id=0)
assert x.tolist()==[[1,2,3],[4,0,0]]
assert y.tolist()==[[2,3,0],[0,-100,-100]]
a,b=instruction_collate([[1,2],[3,4]],pad_id=9)
assert b.tolist()==[[2,9],[4,9]]
print("LLM15 공개 확인 통과 — 설명 답변도 작성하세요.")


## LLM16 · 확장 · DPO 선호 손실 — 5점

**출처:** [Chapter_7_Exercise_Follow_Instructions_dpo.ipynb](../../llm_hands_on/Chapter_7_Exercise_Follow_Instructions_dpo.ipynb) 셀 1 / compute_dpo_loss

**평가 축: 학습 목표·예측값 연결.** Direct Preference Optimization(직접 선호 최적화)의 로그 확률이 이미 계산되어 있습니다.
정책 모델의 chosen−rejected 차이에서 reference의 같은 차이를 빼고, `-logsigmoid(beta × 차이)`의 배치 평균을 반환하세요.
네 입력은 모두 `(B,)`, reference 입력은 gradient 없는 고정 값입니다. 반환은 scalar입니다.
Chapter 7 확장 문제이며 reward model을 따로 만드는 구현은 요구하지 않습니다.

**짧은 설명 문제:** chosen의 상대적 로그 확률이 올라갈 때 이 loss가 작아져야 하는 이유를 적으세요.


In [ ]:
def preference_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta):
    policy_ratio = ???? - ????
    ref_ratio = ???? - ????
    return -F.????(beta * (policy_ratio - ref_ratio)).mean()


**내 설명:** (여기에 이유와 shape를 적으세요.)

**도움 / 걸린 시간:** 없음 / ___분


In [ ]:
a=torch.tensor([-2.,-3.],requires_grad=True)
r=torch.tensor([-4.,-4.])
baseline=preference_loss(a,r,a.detach(),r,.1)
torch.testing.assert_close(baseline,torch.tensor(math.log(2)))
better=preference_loss(a+1,r,a.detach(),r,.1)
assert better < baseline
baseline.backward()
assert torch.all(a.grad < 0)
print("LLM16 공개 확인 통과 — 설명 답변도 작성하세요.")


## 제출 전 확인

- 모든 답안 셀의 빈칸과 NotImplementedError를 해결했나요?
- 새 커널에서 공통 준비→답안→확인 셀 전체가 실행되나요?
- 이유·shape 답변과 도움 사용 여부를 적었나요?
- 틀린 문제 번호와 오류 메시지를 아래에 남겨 주세요. 다음 복습에서는 그 구간만 인자·크기를 바꿔 다시 풉니다.

**미완료 / 질문:**

**총 풀이 시간:**

**참고한 자료 / 힌트:**

원본 노트북에 작성한 설명과 코드를 함께 제출하면 문제 조건에 따라 채점합니다.
공개 확인을 통과해도 독립 풀이·다음날 재현은 별도로 확인합니다.
